In [1]:
import torch
import transformers
print("Torch version:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

c:\Users\sthem\anaconda3\envs\swin_transformer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.5.1+cu121
CUDA disponível: True


In [2]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import random

# Caminhos para os dados
DATASET_PATH = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\cell_images"
INFECTED_PATH = os.path.join(DATASET_PATH, "Parasitized")  # Células infectadas
UNINFECTED_PATH = os.path.join(DATASET_PATH, "Uninfected")  # Células saudáveis

# Transformações de imagem para ResNet-50
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Tamanho esperado pelo Swin Transformer
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Criar dataset personalizado
class MalariaDataset(Dataset):
    def __init__(self, data_path, transform=None):
        self.data_path = data_path
        self.transform = transform
        self.images = []
        self.labels = []
        
        # Carregar imagens infectadas (1)
        for img_name in os.listdir(INFECTED_PATH):
            self.images.append(os.path.join(INFECTED_PATH, img_name))
            self.labels.append(1)
        
        # Carregar imagens não infectadas (0)
        for img_name in os.listdir(UNINFECTED_PATH):
            self.images.append(os.path.join(UNINFECTED_PATH, img_name))
            self.labels.append(0)

        # Embaralhar os dados
        temp = list(zip(self.images, self.labels))
        random.shuffle(temp)
        self.images, self.labels = zip(*temp)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

# Criar instâncias do dataset
dataset = MalariaDataset(DATASET_PATH, transform=transform)

# Dividir em treino e teste (80% treino, 20% teste)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Tamanho do dataset: {len(dataset)}")
print(f"Imagens de treino: {len(train_dataset)}, Imagens de teste: {len(test_dataset)}")


Tamanho do dataset: 27558
Imagens de treino: 22046, Imagens de teste: 5512


In [3]:

from torchvision.models import resnet50, ResNet50_Weights
import torch.nn as nn

# Carregar modelo ResNet-50 pré-treinado no ImageNet
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)

# Usar GPU se disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model_name = "ResNet50"


In [4]:
import os
from PIL import Image

DATASET_PATH = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\cell_images"

# Listar arquivos que podem ser problemáticos
for folder in ["Parasitized", "Uninfected"]:
    folder_path = os.path.join(DATASET_PATH, folder)
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        try:
            with Image.open(file_path) as img:
                img.verify()  # Verifica se a imagem é válida
        except Exception as e:
            print(f"Erro no arquivo: {file_path} - {str(e)}")


In [5]:
for folder in ["Parasitized", "Uninfected"]:
    folder_path = os.path.join(DATASET_PATH, folder)
    for file in os.listdir(folder_path):
        ext = os.path.splitext(file)[-1].lower()
        if ext not in [".png", ".jpg", ".jpeg"]:
            print(f"Arquivo inválido detectado: {file}")

### modelo

In [6]:
from torchvision.models import resnet50, ResNet50_Weights
import torch.nn as nn
import torchvision.models as models

def get_resnet_model():
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 2)  # 2 classes: com e sem malária
    return model


In [14]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np
import os


def train(model, train_loader, val_loader, epochs=5, device='cuda' if torch.cuda.is_available() else 'cpu', model_name='ResNet50', fold=0):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    log_file = "kfold_epoch_results.csv"
    results_list = []


    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        current_lr = optimizer.param_groups[0]['lr']
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            
        scheduler.step()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

        # Avaliação no conjunto de validação com métricas
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        accuracy = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
        precision = precision_score(all_labels, all_preds, zero_division=0)
        recall = recall_score(all_labels, all_preds, zero_division=0)
        f1 = f1_score(all_labels, all_preds, zero_division=0)
        cm = confusion_matrix(all_labels, all_preds)
        tn, fp, fn, tp = cm.ravel()


        # Salvar resultados em um CSV
        results_list.append({
            "fold": fold,
            "epoch": epoch + 1,
            "model": model_name,
            "loss": running_loss / len(train_loader),
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "true_positives": tp,
            "false_positives": fp,
            "true_negatives": tn,
            "false_negatives": fn,
            "learning_rate": current_lr  # log do lr
        })

    torch.save(model.state_dict(), f"resnet_fold{fold}.pth")

    results_df = pd.DataFrame(results_list)
    if os.path.exists(log_file):
        results_df.to_csv(log_file, mode='a', header=False, index=False)
    else:
        results_df.to_csv(log_file, index=False)

'''
# K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n----- Fold {fold + 1} -----")

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

    model = get_resnet_model()
    model_name = f"ResNet50_Fold{fold+1}"
    train(model, train_loader, val_loader, model_name=model_name, epochs=30)
'''


'\n# K-Fold\nskf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)\n\nfor fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):\n    print(f"\n----- Fold {fold + 1} -----")\n\n    train_subset = Subset(dataset, train_idx)\n    val_subset = Subset(dataset, val_idx)\n\n    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)\n    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)\n\n    model = get_resnet_model()\n    model_name = f"ResNet50_Fold{fold+1}"\n    train(model, train_loader, val_loader, model_name=model_name, epochs=30)\n'

In [15]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch.utils.data import Subset, DataLoader

# 1. Extrai os rótulos do dataset original
labels = [dataset[i][1] for i in range(len(dataset))]

# 2. Divide em 80% treino+validação e 20% teste
train_val_indices, test_indices = train_test_split(
    list(range(len(labels))),
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# 3. Cria os Subsets
train_val_dataset = Subset(dataset, train_val_indices)
test_dataset = Subset(dataset, test_indices)

# 4. Cria o DataLoader do conjunto de teste (fixo)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 5. Prepara os rótulos de treino/validação para o StratifiedKFold
train_val_labels = [dataset[i][1] for i in train_val_indices]

# 6. Configura o K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 7. Inicia o loop de treinamento por fold
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_val_dataset)), train_val_labels)):
    print(f"\n----- Fold {fold + 1} -----")

    train_subset = Subset(train_val_dataset, train_idx)
    val_subset = Subset(train_val_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

    model = get_resnet_model()
    model_name = f"ResNet50_Fold{fold+1}"
    train(model, train_loader, val_loader, model_name=model_name, epochs=20, fold=fold+1)



----- Fold 1 -----
Epoch [1/20], Loss: 0.1304
Epoch [2/20], Loss: 0.0727
Epoch [3/20], Loss: 0.0541
Epoch [4/20], Loss: 0.0405
Epoch [5/20], Loss: 0.0319
Epoch [6/20], Loss: 0.0235
Epoch [7/20], Loss: 0.0173
Epoch [8/20], Loss: 0.0168
Epoch [9/20], Loss: 0.0167
Epoch [10/20], Loss: 0.0115
Epoch [11/20], Loss: 0.0030
Epoch [12/20], Loss: 0.0008
Epoch [13/20], Loss: 0.0006
Epoch [14/20], Loss: 0.0003
Epoch [15/20], Loss: 0.0002
Epoch [16/20], Loss: 0.0002
Epoch [17/20], Loss: 0.0001
Epoch [18/20], Loss: 0.0001
Epoch [19/20], Loss: 0.0001
Epoch [20/20], Loss: 0.0003

----- Fold 2 -----
Epoch [1/20], Loss: 0.1349
Epoch [2/20], Loss: 0.0736
Epoch [3/20], Loss: 0.0533
Epoch [4/20], Loss: 0.0365
Epoch [5/20], Loss: 0.0312
Epoch [6/20], Loss: 0.0219
Epoch [7/20], Loss: 0.0204
Epoch [8/20], Loss: 0.0136
Epoch [9/20], Loss: 0.0091
Epoch [10/20], Loss: 0.0099
Epoch [11/20], Loss: 0.0052
Epoch [12/20], Loss: 0.0007
Epoch [13/20], Loss: 0.0006
Epoch [14/20], Loss: 0.0002
Epoch [15/20], Loss: 0.000

### testando no dataset de teste

In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import pandas as pd
import numpy as np
import torch.nn as nn
import os

criterion = nn.CrossEntropyLoss()

def evaluate(model, test_loader, device='cuda' if torch.cuda.is_available() else 'cpu', fold=None, model_name='SwinTransformer', epoch=None, log_file="kfold_epoch_results.csv"):
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(test_loader)
    accuracy = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
    precision = precision_score(all_labels, all_preds, zero_division=0, average='binary')
    recall = recall_score(all_labels, all_preds, zero_division=0, average='binary')
    f1 = f1_score(all_labels, all_preds, zero_division=0, average='binary')

    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()

    print(f"\nAvaliação do modelo {model_name} - Fold {fold} - Epoch {epoch}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    print(f"TP: {tp} | FP: {fp} | TN: {tn} | FN: {fn}")

    report = classification_report(all_labels, all_preds, digits=4)
    print("\n Classification Report:\n")
    print(report)

    result = {
        "fold": fold,
        "epoch": epoch,
        "model": model_name,
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "true_positives": tp,
        "false_positives": fp,
        "true_negatives": tn,
        "false_negatives": fn
    }

    results_df = pd.DataFrame([result])
    if os.path.exists(log_file):
        results_df.to_csv(log_file, mode='a', header=False, index=False)
    else:
        results_df.to_csv(log_file, index=False)


In [17]:
evaluate(model, test_loader, device, fold=6, model_name="ResNet50_Test", epoch=1)


Avaliação do modelo ResNet50_Test - Fold 6 - Epoch 1
Accuracy: 97.22%
Precision: 0.9755 | Recall: 0.9688 | F1: 0.9721
TP: 2670 | FP: 67 | TN: 2689 | FN: 86

 Classification Report:

              precision    recall  f1-score   support

           0     0.9690    0.9757    0.9723      2756
           1     0.9755    0.9688    0.9721      2756

    accuracy                         0.9722      5512
   macro avg     0.9723    0.9722    0.9722      5512
weighted avg     0.9723    0.9722    0.9722      5512

